# Fine-tuning YOLO26n для детекции автомобильных номерных знаков

**Датасет:** AUTO.RIA Numberplate Dataset (ria-com / nomeroff-net), подвыборка ~4000 изображений.

**Модель:** YOLO26n — последний релиз Ultralytics (янв 2026), NMS-free, на 43% быстрее на CPU.

**Почему fine-tune, а не с нуля:** за 2 дня обучить YOLO с нуля на ~4K изображений нереально. Transfer learning с весов COCO даёт качественную детекцию за 15–25 минут на RTX 3050.

**Связь с курсом:** подход «baseline → improved» из ДЗ13–15: сначала берём pretrained YOLO как baseline, измеряем метрики, дообучаем на номерах → улучшенная модель.

In [ ]:
import sys, os, yaml
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import torch
from ultralytics import YOLO

print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

CFG = yaml.safe_load(open('../config.yaml', encoding='utf-8'))
CFG

## 1. Baseline — pretrained YOLO26n без fine-tune

Сначала прогоним pretrained модель на валидации, чтобы зафиксировать baseline (будет плохо — COCO не знает про номера). Это даст нам точку отсчёта, чтобы в отчёте показать, насколько fine-tuning помог.

In [ ]:
data_yaml = Path(CFG['yolo_dataset_dir']) / 'data.yaml'
print('Dataset config:', data_yaml)

baseline = YOLO('yolo26n.pt')
# Примечание: pretrained COCO-YOLO не знает класс license_plate,
# поэтому mAP ожидаемо будет ~0. Это и есть точка отсчёта.
print('Classes COCO:', baseline.names[:5], '...', baseline.names[-3:])

## 2. Fine-tuning на UA номерах

In [ ]:
model = YOLO('yolo26n.pt')

results = model.train(
    data=str(data_yaml),
    epochs=CFG['detector']['epochs'],
    imgsz=CFG['detector']['imgsz'],
    batch=CFG['detector']['batch'],
    device=CFG['detector']['device'],
    project='../runs/detect',
    name='ua_plates_yolo26n',
    patience=10,           # ранняя остановка
    save=True,
    plots=True,
    # Лёгкая аугментация (инсайт из ДЗ14 — aug помогает против overfitting):
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.3,
    degrees=5, translate=0.1, scale=0.3,
    fliplr=0.0,            # для номеров зеркалка ломает текст
    mosaic=1.0, mixup=0.0,
)

## 3. Метрики на val — ключевое для отчёта

Фиксируем mAP50, mAP50-95, precision, recall. Эти цифры идут в пункт «верификация» плана и в 04_evaluation.ipynb.

In [ ]:
metrics = model.val(data=str(data_yaml))
print(f"mAP50     : {metrics.box.map50:.4f}")
print(f"mAP50-95  : {metrics.box.map:.4f}")
print(f"Precision : {metrics.box.mp:.4f}")
print(f"Recall    : {metrics.box.mr:.4f}")

## 4. Сохранить лучшие веса

In [ ]:
import shutil
best = Path('../runs/detect/ua_plates_yolo26n/weights/best.pt')
target = Path('../models/yolo26_plate.pt')
target.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(best, target)
print('Saved to', target, f'({target.stat().st_size/1e6:.1f} MB)')

## 5. Быстрый визуальный sanity-check

In [ ]:
import cv2, matplotlib.pyplot as plt, random
from pathlib import Path

trained = YOLO(str(target))
val_dir = Path(CFG['yolo_dataset_dir']) / 'images/val'
samples = random.sample(list(val_dir.glob('*.jpg')) + list(val_dir.glob('*.png')), k=6)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.ravel(), samples):
    res = trained.predict(str(img_path), conf=0.25, verbose=False)[0]
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    for box in res.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)
    ax.imshow(img); ax.axis('off'); ax.set_title(img_path.name)
plt.tight_layout(); plt.show()